In [2]:
import sys, os, glob
from google.colab import drive
drive.mount('/content/drive')

UTENTE = 'PaoloniCoder'
BRANCH = 'main'

!rm -rf /content/MozartAI
!git clone -b {BRANCH} https://github.com/{UTENTE}/MozartAI.git /content/MozartAI
!pip -q install gradio==5.50.0 pretty_midi

# Cerco i file nel repo, ovunque si trovino, e metto le loro cartelle nel percorso di Python
necessari = ['app.py', 'strumenti.py', 'pipeline.py', 'preprocessing.py', 'model.py', 'midi_convert.py', 'audio.py']
percorsi = {}
for nome in necessari:
    trovati = glob.glob(f'/content/MozartAI/**/{nome}', recursive=True)
    percorsi[nome] = trovati[0] if trovati else None
    if trovati:
        cartella = os.path.dirname(trovati[0])
        if cartella not in sys.path:
            sys.path.insert(0, cartella)

mancanti = [n for n, p in percorsi.items() if p is None]
print("Mancanti:", mancanti if mancanti else "nessuno")
assert not mancanti, "Questi file non sono nel repo (branch main): hai fatto commit e push?"
print({n: p.replace('/content/MozartAI/', '') for n, p in percorsi.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/MozartAI'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 174 (delta 9), reused 2 (delta 2), pack-reused 152 (from 1)
Receiving objects: 100% (174/174), 11.35 MiB | 21.06 MiB/s, done.
Resolving deltas: 100% (76/76), done.
Mancanti: nessuno
{'app.py': 'app.py', 'strumenti.py': 'src/strumenti.py', 'pipeline.py': 'src/pipeline.py', 'preprocessing.py': 'src/preprocessing.py', 'model.py': 'src/model.py', 'midi_convert.py': 'src/midi_convert.py', 'audio.py': 'src/audio.py'}


In [3]:
import importlib.util

os.environ['MOZART_MODELLO'] = '/content/drive/MyDrive/MozartAI/checkpoints_v2/best.pt'

spec = importlib.util.spec_from_file_location('mozart_app', percorsi['app.py'])
mozart_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mozart_app)

print("Modello:", mozart_app.precarica())
demo = mozart_app.crea_app()
demo.queue(max_size=8).launch(share=True, debug=True)

Modello: /content/drive/MyDrive/MozartAI/checkpoints_v2/best.pt


/content/MozartAI/app.py:226: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=tema, css=CSS, title="MozartAI") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fd94cddad7f6a830ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fd94cddad7f6a830ef.gradio.live
